kernel : Python (Pixi)

In [ ]:
import anndata
import gc
import os
import pandas as pd
from anndata import AnnData
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir

anndata.settings.allow_write_nullable_strings = True

raw_count_matrix_location = find_env_dir("RAW_COUNT_MATRIX")
h5ad_matrix_location = find_env_dir("H5AD_MATRIX")

def load_rds_data(rds_path: str, gz: bool, series: str) -> AnnData:
    import anndata2ri
    from rpy2.robjects import r
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects.packages import importr

    importr("base")
    importr("Matrix")
    importr("Seurat")
    importr("SingleCellExperiment")
    importr("monocle3")
    
    if gz:
        read_cmd = f'readRDS(gzcon(gzfile("{rds_path}", "rb")))'
    else:
        read_cmd = f'readRDS("{rds_path}")'

    rcode = f"""
    obj <- {read_cmd}
    if (inherits(obj, "Seurat")) {{
        obj <- as.SingleCellExperiment(obj)
    }}

    if (!inherits(obj, "SingleCellExperiment")) {{
        stop("Loaded object is not a SingleCellExperiment (or derived from it).")
    }}

    anames <- assayNames(obj)
    use_assay <- if ("counts" %in% anames) "counts" else anames[[1]]

    mat <- assay(obj, use_assay)
    mat <- as(mat, "dgCMatrix")

    assay(obj, "counts") <- mat
    # Overwrite 'X' assay with 'counts' assay
    assay(obj, "X") <- assay(obj, "counts")
    obj <- as(obj, "SingleCellExperiment") # Ensure the object is a SingleCellExperiment
    obj
    """
    with localconverter(anndata2ri.converter):
        adata = r(rcode)
        r("if (exists('obj')) rm(obj)")
        r("gc()")

    if not isinstance(adata, AnnData):
        raise ValueError(f"Result is not AnnData. Got type: {type(adata)}")

    adata.layers.pop("counts", None)
    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)
    
    # Multidimensional representations (obsm, varm, layers) are intentionally cleared at this stage,
    # since integrated multi-modal / multi-dimensional analyses will be performed after dataset integration, 
    # overwriting any existing representations.
    adata.obsm = {}
    adata.varm = {}
    adata.layers = {}
    gc.collect()

    adata.obs["series"] = series
    return adata

In [2]:
FILE = "GSE304067_so_clean_paanib1_geo.rds"
SERIES_NAME = "Mace"
SAVE_FILE_NAME = "Mace.h5ad"
adata = load_rds_data(os.path.join(raw_count_matrix_location, SERIES_NAME, FILE), gz = True, series = SERIES_NAME)
adata.obs.head() #type: ignore


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    

R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: Layer ‘scale.data’ is empty 
  


,orig.ident,nCount_RNA,nFeature_RNA,Sample,cell_barcode,sum,detected,percent.top_20,subsets_Mito_sum,subsets_Mito_detected,...,treatment,condition,score,group,batch,nCount_SCT,nFeature_SCT,cell_type_clean,ident,series
AAACAAGCAGCTCGAGACTTTAGG-1,SeuratProject,3611.0,2127,EAE_C831_A5,AAACAAGCAGCTCGAGACTTTAGG-1,3611.0,2127,9.831072,11.0,9,...,C831,EAE,2.5,EAE_C831,1,3760.0,2122,Peripheral_Myeloid,Peripheral_Myeloid,Mace
AAACAAGCAGTTATCCACTTTAGG-1,SeuratProject,3497.0,1999,EAE_C831_A5,AAACAAGCAGTTATCCACTTTAGG-1,3497.0,1999,14.755505,2.0,2,...,C831,EAE,2.5,EAE_C831,1,3729.0,1997,Peripheral_Myeloid,Peripheral_Myeloid,Mace
AAACAAGCATACCTTAACTTTAGG-1,SeuratProject,3313.0,2124,EAE_C831_A5,AAACAAGCATACCTTAACTTTAGG-1,3313.0,2124,7.093269,1.0,1,...,C831,EAE,2.5,EAE_C831,1,3636.0,2124,Microglia,Microglia,Mace
AAACCAATCAAGTTTCACTTTAGG-1,SeuratProject,3657.0,2108,EAE_C831_A5,AAACCAATCAAGTTTCACTTTAGG-1,3657.0,2108,10.336341,0.0,0,...,C831,EAE,2.5,EAE_C831,1,3805.0,2107,Microglia,Microglia,Mace
AAACCAATCACCACACACTTTAGG-1,SeuratProject,3080.0,1965,EAE_C831_A5,AAACCAATCACCACACACTTTAGG-1,3080.0,1965,8.311688,0.0,0,...,C831,EAE,2.5,EAE_C831,1,3575.0,1958,Peripheral_Myeloid,Peripheral_Myeloid,Mace


In [16]:
assert isinstance(adata.obs, pd.DataFrame)
adata.obs["sample"] = adata.obs["Sample"]
adata.obs["treatment"] = adata.obs["treatment"]
adata.obs["condition"] = adata.obs["condition"]
adata.obs["donor"] = adata.obs["mouse"]
adata.obs["sex"] = adata.obs["sex"]
adata.obs["score"] = adata.obs["score"]
adata.obs["group"] = adata.obs["group"]
adata.obs["pool"] = adata.obs["batch"]
adata.obs["celltype"] = adata.obs["cell_type_clean"]
adata.obs["series"] = adata.obs["series"]


keep = ["sample", "treatment", "condition", "donor", "sex", "score", "group", "pool", "celltype", "series"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [18]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(h5ad_matrix_location, SAVE_FILE_NAME))
del adata
gc.collect()

3769